선형 회귀 모델 -> 숫자형 타깃 변수의 크기 예측 (회귀 문제)
로지스틱 회귀 모델 -> 문자형 타깃 변수의 유형 예측 (분류 문제)

분류 모델은 2단계에 따라 타깃을 예측

1단계 : 각 클래스가 선택될(발생할) 확률을 계산
2단계 : 가장 확률이 높은 클래스를 타깃의 예측값으로 선택

즉, 모델이 직접 예측하는 값은 각 클래스가 될 확률 (클래스 자체 X)

인코딩 : 문자형 변수를 숫자 형태로 변환하는 과정

1. [one-hot encoding] : 각 클래스를 확률 형태의 숫자 벡터로 표현
    -> 클래스 수만큼 열을 만들고 타깃이 해당하는 클래스에는 1, 나머지 클래스에는 0을 표사
    즉, 문자형 변수를 더미 변수로 변환

2. [label encoding] : 각 클래스에 고유한 정숫값을 부여
    -> 구현 쉬움, 메모리 적게 사용

In [1]:
#[실습 04-01]
#one-hot encoding

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score

In [2]:
#[실습 04-02]

dat = {
    '나이'    : [25,33,55],
    '스마트폰' : ['체리','망고','키위']
}

df = pd.DataFrame(dat)
df

,나이,스마트폰
0,25,체리
1,33,망고
2,55,키위


In [ ]:
#[실습 04-03]

y_oh = pd.get_dummies(df["스마트폰"]).astype(int)

print("원핫 인코딩 결과")
print(y_oh)
# 가나다순 정렬

원핫 인코딩 결과
   망고  체리  키위
0   0   1   0
1   1   0   0
2   0   0   1


In [4]:
#[실습 04-04]
#label encoding

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_idx = le.fit_transform(df['스마트폰'])

print("라벨 인코딩 결과")
print(y_idx)

라벨 인코딩 결과
[1 0 2]


로지스틱 회귀 모델 => 클래스별 확률이 계산되는 과정

1.  각 클래스마다 가중치(w)와 편향(b)을 설정
    입력 변수가 1개인 경우 클래스마다 하나의 가중치와 편향을 사용
    w_k는 클래스 k의 가중치, b_k는 클래스 k의 편향

2.  각 클래스 k에 대해 입력 변수 X의 선형 함수로 실숫값 z_k를 계산

    z_0 = (w_0 * X) + b_0
    z_1 = (w_1 * X) + b_1
    z_2 = (w_2 * X) + b_2

이렇게 계산된 z = [z_0,z_1,z_2]는 각 클래스에 대한 score (확률 X)

3.  클래스별 점수 z_0,z_1,z_2를 Softmax 함수를 사용하여 클래스별 확률 p_0,p_1,p_2로 변환
    p_0 = e^z_0 / (e^z_0 + e^z_1 + e^z_2)
    ...
    

In [6]:
#[실습 04-05]

# 입력 변수 지정
X = df[['나이']]

# 클래스별 파라미터 지정
w = [-0.05, 0.02, 0.01]
b = [0,0,0]

# 선형변환
z0 = w[0]*X + b[0]
z1 = w[1]*X + b[1]
z2 = w[2]*X + b[2]

# 확률 계산
prob = pd.DataFrame([]) # 초기화
prob['망고'] = np.exp(z0) / (np.exp(z0)+np.exp(z1)+np.exp(z2))
prob['체리'] = np.exp(z1) / (np.exp(z0)+np.exp(z1)+np.exp(z2))
prob['키위'] = np.exp(z2) / (np.exp(z0)+np.exp(z1)+np.exp(z2))

print("클래스별 예측 확률");
prob


클래스별 예측 확률


,망고,체리,키위
0,0.088997,0.512144,0.398858
1,0.054594,0.549999,0.395407
2,0.013315,0.625692,0.360993


분류 문제에서는 모델이 예측한 클래스별 확률의 정확도 평가를 위해
Cross Entropy, CE
라는 손실 함수를 사용
:   모델이 실제 타깃 클래스의 확률을 얼마나 높게 예측했는지를 하나의 숫자로 계산
    모델이 실제 클래스의 확률을 1에 가깝게 예측할수록 CE는 작아짐
    잘못된 예측에 Penalty를 주는 손실 함수